<a href="https://colab.research.google.com/github/salihnacar/aerial_threat_surveillance/blob/main/ImageProcessing_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio
!pip install opencv-python
from skimage import img_as_float, filters, segmentation,feature


In [ ]:
import gradio as gr
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from skimage import morphology, color
import cv2

def image_info(image):
    image_array = np.array(image)
    image_shape = f"Shape: {image_array.shape}"

    # Grayscale Image
    grayscale_image = image.convert('L')

    # Binary (Black and White) Image
    _, binary_image = cv2.threshold(np.array(grayscale_image), 128, 255, cv2.THRESH_BINARY)
    binary_image = Image.fromarray(binary_image)

    # HSV Image
    hsv_image = cv2.cvtColor(image_array, cv2.COLOR_RGB2HSV)
    hsv_image = Image.fromarray(hsv_image, 'HSV')

    # YCbCr Image
    ycbcr_image = cv2.cvtColor(image_array, cv2.COLOR_RGB2YCrCb)
    ycbcr_image = Image.fromarray(ycbcr_image, 'YCbCr')

    # LAB Image
    lab_image = cv2.cvtColor(image_array, cv2.COLOR_RGB2LAB)
    lab_image = Image.fromarray(lab_image, 'LAB')

    # Indexed (Palette-based) Image
    indexed_image = image.convert('P', palette=Image.ADAPTIVE)

    # Sepia Image
    sepia_image = np.array(image)
    sepia_filter = np.array([[0.393, 0.769, 0.189],
                             [0.349, 0.686, 0.168],
                             [0.272, 0.534, 0.131]])
    sepia_image = cv2.transform(sepia_image, sepia_filter)
    sepia_image = np.clip(sepia_image, 0, 255)
    sepia_image = Image.fromarray(sepia_image.astype(np.uint8))

    # Negative Image
    negative_image = Image.fromarray(255 - image_array)

    # High Dynamic Range (HDR) Image (simple emulation)
    hdr_image = cv2.detailEnhance(image_array, sigma_s=12, sigma_r=0.15)
    hdr_image = Image.fromarray(hdr_image)

    # Channel-separated Images
    red_channel = Image.fromarray(image_array[:, :, 0], 'L')
    green_channel = Image.fromarray(image_array[:, :, 1], 'L')
    blue_channel = Image.fromarray(image_array[:, :, 2], 'L')

    pixel_data_matrix = str(image_array)

    return (image_shape, grayscale_image, binary_image, hsv_image, ycbcr_image, lab_image, indexed_image, sepia_image,
            negative_image, hdr_image, red_channel, green_channel, blue_channel, pixel_data_matrix)


def calculate_histogram(image):
    image_array = np.array(image.convert('L'))
    height, width = image_array.shape
    histogram = np.zeros(256)
    for i in range(height):
        for j in range(width):
            histogram[image_array[i, j]] += 1
    plt.figure()
    plt.bar(range(256), histogram, width=1.0, edgecolor='black')
    plt.title("Image Histogram")
    plt.xlabel("Pixel value")
    plt.ylabel("Frequency")
    plt.savefig("histogram.png")
    plt.close()
    return "histogram.png"

def histogram_equalization(image, start, end):
    image_array = np.array(image.convert('L'))
    height, width = image_array.shape
    histogram = np.zeros(256)
    for i in range(height):
        for j in range(width):
            histogram[image_array[i, j]] += 1
    cdf = np.cumsum(histogram)
    cdf_normalized = (cdf - cdf[start]) / (cdf[end] - cdf[start]) * 255
    cdf_normalized = np.clip(cdf_normalized, 0, 255).astype(np.uint8)
    equalized_image = cdf_normalized[image_array]
    equalized_pil = Image.fromarray(equalized_image)
    return equalized_pil

def convolve(image, kernel):
    image = image.astype(np.float32)
    kernel_height, kernel_width = kernel.shape
    image_height, image_width, num_channels = image.shape
    pad_height = kernel_height // 2
    pad_width = kernel_width // 2
    padded_image = np.pad(image, ((pad_height, pad_height), (pad_width, pad_width), (0, 0)), mode='constant', constant_values=0)
    output_image = np.zeros_like(image)
    for y in range(image_height):
        for x in range(image_width):
            for c in range(num_channels):
                region = padded_image[y:y + kernel_height, x:x + kernel_width, c]
                output_image[y, x, c] = np.sum(region * kernel)
    output_image = np.clip(output_image, 0, 255)
    return output_image.astype(np.uint8)

kernels = {
    "Blurring": np.array([[1, 1, 1],
                          [1, 1, 1],
                          [1, 1, 1]]) / 8.0,
    "Sharpening": np.array([[0, -1, 0],
                            [-1, 5, -1],
                            [0, -1, 0]]),
    "Edge Detection": np.array([[-1, -1, -1],
                                [-1, 8, -1],
                                [-1, -1, -1]])
}



def region_growing(image, seed_point, threshold=10):
    seed_row, seed_col = int(seed_point[1]), int(seed_point[0])
    image_gray = np.array(image.convert('L'))
    if seed_row < 0 or seed_row >= image_gray.shape[0] or seed_col < 0 or seed_col >= image_gray.shape[1]:
        raise ValueError("Seed point is out of bounds")
    mask = morphology.flood(image_gray, (seed_row, seed_col), tolerance=threshold)
    region = np.zeros_like(image_gray, dtype=np.uint8)
    region[mask] = 0
    colored_region = color.gray2rgb(region)
    result_image = image.convert('RGB')
    result_image_np = np.array(result_image)
    result_image_np[mask] = colored_region[mask]
    return Image.fromarray(result_image_np.astype(np.uint8))



def apply_sobel(image, orientation):
    image_gray = np.array(image.convert('L'))
    if orientation == "Horizontal":
        sobelx = cv2.Sobel(image_gray, cv2.CV_64F, 1, 0, ksize=3)
        sobelx = np.absolute(sobelx)
        sobelx = np.uint8(sobelx)
        return Image.fromarray(sobelx)
    elif orientation == "Vertical":
        sobely = cv2.Sobel(image_gray, cv2.CV_64F, 0, 1, ksize=3)
        sobely = np.absolute(sobely)
        sobely = np.uint8(sobely)
        return Image.fromarray(sobely)
    else:
        sobelx = cv2.Sobel(image_gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(image_gray, cv2.CV_64F, 0, 1, ksize=3)
        sobel = np.hypot(sobelx, sobely)
        sobel = np.uint8(sobel)
        return Image.fromarray(sobel)
def apply_canny(image, threshold):
    image_gray = np.array(image.convert('L'))
    edges = cv2.Canny(image_gray, threshold, threshold * 2)
    return Image.fromarray(edges)

def apply_filter(image, filter_type, orientation=None, threshold=None):
    if filter_type == "Sobel":
        return apply_sobel(image, orientation)
    elif filter_type == "Canny":
        return apply_canny(image, threshold)
    else:
        kernel = kernels.get(filter_type)
        if kernel is None:
            raise ValueError("Unknown filter type")
        return Image.fromarray(convolve(np.array(image), kernel))
def active_contours(image, alpha=0.015, beta=10, gamma=0.001, max_iterations=2500, canny_sigma=1):
    image_gray = np.array(image.convert('L'))
    edges = feature.canny(image_gray, sigma=canny_sigma)
    edges_dilated = filters.gaussian(edges, sigma=2) > 0.1  # Dilate edges

    init = np.array(np.nonzero(edges_dilated)).T  # Find initial contours

    snake = segmentation.active_contour(filters.gaussian(image_gray, 3),
                                        init, alpha=alpha, beta=beta, gamma=gamma, max_iterations=max_iterations, coordinates='rc')

    fig, ax = plt.subplots()
    ax.imshow(image_gray, cmap=plt.cm.gray)
    ax.plot(init[:, 1], init[:, 0], '--r', lw=1)
    ax.plot(snake[:, 1], snake[:, 0], '-b', lw=2)
    ax.set_title('Active Contours')
    plt.axis('off')
    plt.savefig("active_contours.png")
    plt.close()
    return "active_contours.png"




def capture_point(evt: gr.SelectData):
    return [evt.index[1], evt.index[0]]


with gr.Blocks() as demo:
    with gr.Tabs():
        with gr.TabItem("Image Properties"):
          gr.Markdown("## Image Properties Display")
          gr.Markdown("Upload an RGB image to see its properties such as shape and sample pixel data.")
          with gr.Row():
              with gr.Column():
                  image_input1 = gr.Image(type="pil", label="Upload Image for Properties")
                  submit_button1 = gr.Button("Submit")
              with gr.Column():
                  image_shape = gr.Textbox(label="Image Shape")
                  with gr.Row():
                      grayscale_image = gr.Image(label="Grayscale Image")
                      binary_image = gr.Image(label="Binary Image")
                  with gr.Row():
                      hsv_image = gr.Image(label="HSV Image")
                      ycbcr_image = gr.Image(label="YCbCr Image")
                  with gr.Row():
                      lab_image = gr.Image(label="LAB Image")
                      indexed_image = gr.Image(label="Indexed Image")
                  with gr.Row():
                      sepia_image = gr.Image(label="Sepia Image")
                      negative_image = gr.Image(label="Negative Image")
                  with gr.Row():
                      hdr_image = gr.Image(label="HDR Image")
                  with gr.Row():
                      red_channel = gr.Image(label="Red Channel")
                      green_channel = gr.Image(label="Green Channel")
                      blue_channel = gr.Image(label="Blue Channel")
                  pixel_data_matrix = gr.Textbox(label="Pixel Data Matrix", lines=20, interactive=True)

          submit_button1.click(image_info, inputs=image_input1, outputs=[image_shape, grayscale_image, binary_image, hsv_image,
                                                                        ycbcr_image, lab_image, indexed_image, sepia_image,
                                                                        negative_image, hdr_image, red_channel, green_channel,
                                                                        blue_channel, pixel_data_matrix])



        with gr.TabItem("Histogram"):
            gr.Markdown("## Histogram Plot")
            gr.Markdown("Upload an RGB image to see its histogram.")
            with gr.Row():
                with gr.Column():
                    image_input2 = gr.Image(type="pil", label="Upload Image for Histogram")
                    submit_button2 = gr.Button("Submit")
                with gr.Column():
                    histogram_image = gr.Image(type="pil", label="Histogram")
            submit_button2.click(calculate_histogram, inputs=image_input2, outputs=histogram_image)

        with gr.TabItem("Histogram Equalization"):
            gr.Markdown("## Histogram Equalization")
            gr.Markdown("Upload an RGB image and adjust the start and end range for histogram equalization.")
            with gr.Row():
                with gr.Column():
                    image_input3 = gr.Image(type="pil", label="Upload Image for Equalization")
                    start_range = gr.Slider(0, 255, step=1, label="Start Range")
                    end_range = gr.Slider(0, 255, step=1, label="End Range")
                    submit_button3 = gr.Button("Submit")
                with gr.Column():
                    equalized_image = gr.Image(type="pil", label="Equalized Image")
            submit_button3.click(histogram_equalization, inputs=[image_input3, start_range, end_range], outputs=equalized_image)

        with gr.TabItem("Image Filtering"):
            gr.Markdown("## Image Filtering")
            gr.Markdown("Upload an RGB image and select a filter to apply.")
            with gr.Row():
                with gr.Column():
                    image_input4 = gr.Image(type="pil", label="Upload Image for Filtering")
                    filter_type = gr.Radio(choices=["Blurring", "Sharpening", "Edge Detection", "Sobel", "Canny"], label="Choose Filter")
                    orientation = gr.Radio(choices=["Horizontal", "Vertical", "Both"], label="Orientation", visible=False)
                    threshold = gr.Slider(0, 255, step=1, label="Threshold", value=100, visible=False)
                    submit_button4 = gr.Button("Submit")
                with gr.Column():
                    filtered_image = gr.Image(type="pil", label="Filtered Image")

            def update_filter_options(filter_type):
                if filter_type == "Sobel":
                    return gr.update(visible=True), gr.update(visible=False)
                elif filter_type == "Canny":
                    return gr.update(visible=False), gr.update(visible=True)
                else:
                    return gr.update(visible=False), gr.update(visible=False)

            filter_type.change(update_filter_options, inputs=filter_type, outputs=[orientation, threshold])
            submit_button4.click(apply_filter, inputs=[image_input4, filter_type, orientation, threshold], outputs=filtered_image)


        with gr.TabItem("Region Growing"):
            gr.Markdown("## Region Growing")
            gr.Markdown("Upload an RGB image, select a point, and apply region growing on it.")
            with gr.Row():
                with gr.Column():
                    image_input5 = gr.Image(type="pil", label="Upload Image for Region Growing", interactive=True)
                    threshold = gr.Slider(0, 255, step=1, label="Threshold", value=10)
                    submit_button5 = gr.Button("Submit")
                with gr.Column():
                    pixel_info = gr.Textbox(label="Selected Pixel Info")
                    output_image = gr.Image(type="pil", label="Output Image")

            # Updated function call to include coordinates and show pixel info
            def get_seed_and_process(image, threshold, evt: gr.SelectData):
                x, y = int(evt.index[0]), int(evt.index[1])
                pixel_value = image.getpixel((x, y))
                pixel_info = f"Coordinates: ({x}, {y}), Pixel Value: {pixel_value}"
                processed_image = region_growing(image, (x, y), threshold)
                return pixel_info, processed_image

            image_input5.select(get_seed_and_process, inputs=[image_input5, threshold], outputs=[pixel_info, output_image])

        with gr.TabItem("Active Contours"):
            gr.Markdown("## Active Contours")
            gr.Markdown("Upload an RGB image and apply active contours for segmentation.")
            with gr.Row():
                with gr.Column():
                    image_input6 = gr.Image(type="pil", label="Upload Image for Active Contours")
                    alpha = gr.Slider(0.001, 0.1, step=0.001, label="Alpha", value=0.015)
                    beta = gr.Slider(1, 20, step=1, label="Beta", value=10)
                    gamma = gr.Slider(0.001, 0.01, step=0.001, label="Gamma", value=0.001)
                    max_iterations = gr.Slider(100, 5000, step=100, label="Max Iterations", value=2500)
                    canny_sigma = gr.Slider(0.1, 3.0, step=0.1, label="Canny Sigma", value=1)
                    submit_button6 = gr.Button("Submit")
                with gr.Column():
                    active_contours_image = gr.Image(type="pil", label="Active Contours Image")

            submit_button6.click(active_contours, inputs=[image_input6, alpha, beta, gamma, max_iterations, canny_sigma], outputs=active_contours_image)

# Launch the app
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8fe5168f2d077cf103.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
